In [ ]:
import os
import sys
import time
import pickle
import json
from typing import Tuple, Union, List, Dict, Final

import numpy as np
from PIL import Image
import cv2
import torch
from pycocotools import mask as coco_mask_util

In [ ]:
from tqdm import tqdm

def expand_bbox(
    bbox: Union[List[int], Tuple[int], np.ndarray], 
    image_width: int, 
    image_height: int, 
    percentage_to_expand_bbox_boundaries: float
) -> List[int]:
    
    xtl, ytl, xbr, ybr = bbox
    delta_x: int = int(percentage_to_expand_bbox_boundaries * (xbr - xtl) / 2)
    delta_y: int = int(percentage_to_expand_bbox_boundaries * (ybr - ytl) / 2)
    # expand by one pixel on each side at least to cover boundaries            
    delta_x = max(1, delta_x)
    delta_y = max(1, delta_y)
            
    xmin: int = max(0, xtl - delta_x)
    ymin: int = max(0, ytl - delta_y)
    xmax: int = min(image_width, xbr + delta_x)
    ymax: int = min(image_height, ybr + delta_y)
    
    return [xmin, ymin, xmax, ymax]

def convert_to_coco_api(images_path: str, 
                        annotations_path: str, 
                        id2label: Dict[int, str], # this is for the converted dataset, should start with 0 for YOLO, DETR models (RT/RF), 
                                                  # and 1 for Mask R-CNN/Mask2Fomer
                        instance_segmentation: bool = False,
                        percentage_to_expand_bbox_boundaries: float = 0.0):
    # load all images and annotations
    # the assumption is the image and its annotation use the same name
    imgs = list(sorted(os.listdir(images_path)))
    annotations = list(sorted(os.listdir(annotations_path)))
    start_class_id: int = min(list(id2label.keys()))
    
    if len(imgs) != len(annotations):
        print("[ERROR]: The list of images and masks are not consistent")
        return False, {}
    
    for i, img_filename in enumerate(imgs):
        # drop the image/mask filename extension 
        # (anything after the last '.' in the filename is considered as extension)
        img_name = ".".join(img_filename.strip().split('.')[:-1])
        annots_name = ".".join(annotations[i].strip().split('.')[:-1])
        if img_name != annots_name:
            print("[ERROR]: Inconsistent annotations file :{} found for image file: {}".format(annots_name, img_name))
            return False, {}
    # the index for annotations starts at 1
    annots_id = 1
    categories = set()
    json_annotations = {"images": [], "categories": [], "annotations": []}
    for idx in tqdm(range(len(imgs))):
        # load the image
        img_path = os.path.join(images_path, imgs[idx])
        # read the image, we only read the image to get the size of it
        # so no need to change the format (BGR to RGB) or convert to PIL 
        opencv_img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        image_height, image_width = opencv_img.shape[:2]
        img_dict = {}
        img_dict["id"] = idx + 1
        img_dict["file_name"] = imgs[idx]
        img_dict["height"] = image_height
        img_dict["width"] = image_width
        json_annotations["images"].append(img_dict)


        # annotations
        boxes: List[List[int]] = []
        labels: List[int] = []
        masks: List[np.ndarray] = []
        # load the annotations (masks)
        annots_path = os.path.join(annotations_path, annotations[idx])

        if instance_segmentation:
            # load the annotations
            filehandler = open(annots_path, 'rb')
            annots = pickle.load(filehandler)
            filehandler.close()
            
            for record in annots['annotations']:
                xtl, ytl, xbr, ybr = record['bbox']
                # no need to check the validity 
                if xtl >= xbr or ytl >= ybr:
                    continue
                # subtract 1 - start_class_id to adjust the labels
                # class IDs start from 1 in Mask R-CNN annotations, so if training YOLO/DETR, subtract 1
                # otherwise, leave the labels as is
                labels.append(int(record['category_id'] - 1 + start_class_id))
                mask: np.ndarray = coco_mask_util.decode(record['segmentation'])
                # mask in full image resolution
                full_res_mask: np.ndarray = np.zeros((image_height, image_width), np.uint8)
                full_res_mask[ytl:ybr, xtl:xbr] = mask
                
                masks.append(full_res_mask)
                
                expanded_bbox = expand_bbox(
                    bbox=[xtl, ytl, xbr, ybr],
                    image_width=image_width, 
                    image_height=image_height, 
                    percentage_to_expand_bbox_boundaries=percentage_to_expand_bbox_boundaries
                )
                # will convert bbox to xtl, ytl, w, h format below
                boxes.append(expanded_bbox)
        else:
            with open(annots_path,'r') as annot_file:
                for line in annot_file:
                    fields = line.strip().split(' ')
                    (label, center_x, center_y, w, h) = fields
                    xtl = int((float(center_x) - float(w) / 2.0) * image_width)
                    ytl = int((float(center_y) - float(h) / 2.0) * image_height)
                    xbr = int((float(center_x) + float(w) / 2.0) * image_width)
                    ybr = int((float(center_y) + float(h) / 2.0) * image_height)
                    if xtl >= xbr or ytl >= ybr:
                        continue
                    # convert the label to an integer from string
                    # for YOLO, labels start from 0, adjust the starting class IDs if needed
                    label = int(label + start_class_id) 
                    expanded_bbox: List[int] = expand_bbox(
                        bbox=[xtl, ytl, xbr, ybr],
                        image_width=image_width, 
                        image_height=image_height, 
                        percentage_to_expand_bbox_boundaries=percentage_to_expand_bbox_boundaries
                    )
                    # will convert bbox to xtl, ytl, w, h format below
                    boxes.append(expanded_bbox)
                    labels.append(label)
        
        for i, box in enumerate(boxes):
            record = {}
            record["image_id"] = idx + 1
            record['category_id'] = labels[i] 
            categories.add(record['category_id'])
            if instance_segmentation:
                record["segmentation"] = coco_mask_util.encode(np.asarray(masks[i], order="F"))
                record["segmentation"]['counts'] = record["segmentation"]['counts'].decode('utf8')
            xmin, ymin, xmax, ymax = [int(v) for v in box]
            #convert to xywh
            record['bbox'] = [xmin, ymin, xmax - xmin, ymax - ymin]
            record["area"] = (ymax - ymin) * (xmax - xmin)
            record["iscrowd"] = 0
            record["id"] = annots_id
                
            json_annotations["annotations"].append(record)
            annots_id += 1 
            
    json_annotations["categories"] = [{"id": i, "name": id2label[i], "supercategory": "biology"} for i in sorted(categories)]

    return True, json_annotations

In [ ]:
MASK_RCNN_ANNOTAIONS_FORMAT: bool = True
BASE_PATH = '/home/cellareye/Cellanome/dl-mehdi/Mask RCNN/data/down_sampled_all_4_class_for_DINOv2'
if MASK_RCNN_ANNOTAIONS_FORMAT:
    ANNOTS_FOLDER_NAME = "masks"
else:
    ANNOTS_FOLDER_NAME = "labels"

TRAIN_IMAGE_FOLDER = os.path.join(BASE_PATH, 'images', 'train')
TRAIN_MASK_FOLDER = os.path.join(BASE_PATH, ANNOTS_FOLDER_NAME, 'train')
TEST_IMAGE_FOLDER =os.path.join(BASE_PATH, 'images', 'test')
TEST_MASK_FOLDER = os.path.join(BASE_PATH, ANNOTS_FOLDER_NAME, 'test')


OUTPUT_FOLDER = BASE_PATH
# mapping between the class IDs and class names for the annotated data 
# this is not used anywhere here but included for information
# this the mapping used for generating the parsed data (class IDs) in the annotations
# for MASK_RCNN_ANNOTAIONS_FORMAT set to True, it starts from 1, otherwise starts from 0
LABEL_MAP = {0: 'cell', 1: 'bead', 2: 'soma', 3: 'cell-adhered'}
REVERESE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

In [ ]:
import json

success, json_annots = convert_to_coco_api(images_path=TRAIN_IMAGE_FOLDER, 
                                           annotations_path=TRAIN_MASK_FOLDER, 
                                           id2label=LABEL_MAP,
                                           instance_segmentation = MASK_RCNN_ANNOTAIONS_FORMAT)
with open(os.path.join(OUTPUT_FOLDER, 'train_annotations.json'), 'w') as file:
    json.dump(json_annots, file)

success, json_annots = convert_to_coco_api(images_path=TEST_IMAGE_FOLDER, 
                                           annotations_path=TEST_MASK_FOLDER, 
                                           id2label=LABEL_MAP,
                                           instance_segmentation = MASK_RCNN_ANNOTAIONS_FORMAT)
with open(os.path.join(OUTPUT_FOLDER, 'test_annotations.json'), 'w') as file:
    json.dump(json_annots, file)